# MASLD FIB-4 Study — Exploratory Data Analysis

This notebook provides an interactive space for EDA after the pipeline has run.
Run `define_cohort.py` first to generate `data/nhanes_masld_cohort.parquet`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

with open('../config/config.yaml') as f:
    config = yaml.safe_load(f)

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

## 1. Load MASLD Cohort

In [ ]:
df = pd.read_parquet('../data/nhanes_masld_cohort.parquet')
print(f'Cohort: {len(df):,} rows x {len(df.columns)} columns')
df.head()

## 2. FIB-4 Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['FIB4'].dropna().clip(upper=5), bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(1.30, color='orange', linestyle='--', label='Low/Indeterminate (1.30)')
axes[0].axvline(2.67, color='red', linestyle='--', label='Indeterminate/High (2.67)')
axes[0].set_title('FIB-4 Distribution (MASLD cohort)')
axes[0].legend()

axes[1].hist(df['LUXSMED'].dropna().clip(upper=30), bins=50, color='seagreen', edgecolor='white')
axes[1].axvline(8.0, color='orange', linestyle='--', label='Significant fibrosis (8 kPa)')
axes[1].axvline(12.0, color='red', linestyle='--', label='Advanced fibrosis (12 kPa)')
axes[1].set_title('LSM (FibroScan) Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Misclassification Summary

In [ ]:
print('Misclassification label counts:')
print(df['MISCLASS_LABEL'].value_counts())
print()
low_risk = df[df['FIB4_CAT'] == 0]
fn_rate = (low_risk['FIB4_FALSE_NEGATIVE'] == 1).mean()
print(f'FIB-4 False Negative Rate (low-risk stratum): {fn_rate*100:.1f}%')
print(f'N false negatives: {(low_risk["FIB4_FALSE_NEGATIVE"] == 1).sum():,} / {len(low_risk):,}')

## 4. Race/Ethnicity Distribution

In [ ]:
from src.utils.nhanes_codebook import RACE_LABELS

race_fn = low_risk.groupby('RIDRETH3')['FIB4_FALSE_NEGATIVE'].agg(['sum', 'count'])
race_fn['rate'] = race_fn['sum'] / race_fn['count'] * 100
race_fn['label'] = race_fn.index.map(RACE_LABELS)
race_fn = race_fn.sort_values('rate', ascending=False)

race_fn[['label', 'count', 'sum', 'rate']].rename(
    columns={'sum': 'n_fn', 'count': 'n_total', 'rate': 'fn_rate_%'}
)

## 5. Correlation — FIB-4 Components vs LSM

In [ ]:
corr_vars = ['RIDAGEYR', 'LBXSASSI', 'LBXSATSI', 'LBXPLTSI', 'FIB4',
             'LBXSGTSI', 'LBXGH', 'BMXBMI', 'BMXWAIST', 'LUXSMED']
available = [v for v in corr_vars if v in df.columns]
corr_df = df[available].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.3)
plt.title('Correlation — FIB-4 Components and LSM')
plt.tight_layout()
plt.show()